In [1]:
import os
import polars as pl
import altair as alt

In [2]:
df = pl.read_parquet(os.path.join("data","segments.parquet"))

In [3]:
df.columns

['id',
 'name',
 'activity_type',
 'distance',
 'average_grade',
 'elevation_high',
 'elevation_low',
 'total_elevation_gain',
 'start_latlng',
 'end_latlng',
 'country',
 'state',
 'city',
 'effort_count',
 'athlete_count',
 'date',
 'start_to_finish_distance']

In [4]:
df.filter((pl.col("country") == "Czechia") & (pl.col('distance') > 500) & (pl.col('activity_type') == 'Ride')).select(pl.col('name')).unique().to_series().to_list()

['stupani k vetr. mlynu Kuzelov',
 'Dářská rašeliniště',
 'Lysá hora - 5 km',
 'S&Ch20 Masarykův okruh',
 'Final climb ke srubu na Deštnou',
 'Sprintík křižovatka/skalní mlýn',
 'Červená podél Zlaté stoky do Třeboně',
 'Sprint Páteček - Santos',
 'Novomlýnská 5: Věstonice-Strachotín',
 'Černý důl konec Hoffmanky',
 'Kynšperský ATAK',
 'podle Labe do Týnce',
 'time trial at Svratka',
 'Bunč climb',
 'Zbraslav - Komořany',
 'Z Lednice po silnici k odbocce na Janohrad',
 'Szpic Jested  last 3 km ( od rozc na Dub)',
 'Podle Radbuzy',
 'Risova studanka - od zavory',
 'Nová cyklostezka do Bosche  2',
 'Kopec pod Barborou',
 'Hostyn-od.zastavky.po.znacku.P',
 'Papezov - Prazmo',
 'Šantovka - Kojeňák',
 'Vlevo dole jezero Most',
 'Výstup na Klínovec',
 'svadov—valtirov',
 'Jiráskovo nábřeží k Husovce',
 'Srbsko - Hlásná Třebáň',
 'Zrucskej smrtak',
 'Chrudim climb',
 'Sedmička, Lipno {',
 'Máchovo kritérium - přes bory kolem vrchu Borný',
 'Rokytka od myčky',
 'Mutěnka - z Kyjova po odbočku na

In [5]:
def segment(nazev):
    segm = df.filter(pl.col('name') == nazev).group_by_dynamic("date", every="1d").agg(pl.col("effort_count").max())
    segm = segm.with_columns(pl.col("effort_count").diff().alias("daily_difference"))
    segm = segm.with_columns(pl.when(pl.col("daily_difference") < 0).then(None).otherwise(pl.col("daily_difference")).alias("daily_difference"))
    segm = segm.with_columns(pl.col("date").cast(str)).drop_nulls()
    print(segm)
    return alt.Chart(segm, width=800).mark_bar().encode(
        x='date:O',
        y='daily_difference:Q'
    )

In [6]:
segment("Židlochovice -> Olympia")

shape: (126, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000 ┆ 15827        ┆ 13               │
│ 2024-12-26 00:00:00.000000 ┆ 15841        ┆ 14               │
│ 2024-12-27 00:00:00.000000 ┆ 15857        ┆ 16               │
│ 2024-12-28 00:00:00.000000 ┆ 15874        ┆ 17               │
│ 2024-12-29 00:00:00.000000 ┆ 15888        ┆ 14               │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 17899        ┆ 17               │
│ 2025-04-29 00:00:00.000000 ┆ 17922        ┆ 23               │
│ 2025-04-30 00:00:00.000000 ┆ 17950        ┆ 28               │
│ 2025-05-01 00:00:00.000000 ┆ 18050        ┆ 100              │
│ 2025-05

alt.Chart(...)

In [7]:
segment("Revnicak Eve")

shape: (116, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000 ┆ 43467        ┆ 3                │
│ 2024-12-27 00:00:00.000000 ┆ 43469        ┆ 2                │
│ 2024-12-28 00:00:00.000000 ┆ 43471        ┆ 2                │
│ 2024-12-29 00:00:00.000000 ┆ 43478        ┆ 7                │
│ 2024-12-30 00:00:00.000000 ┆ 43481        ┆ 3                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 44994        ┆ 28               │
│ 2025-04-29 00:00:00.000000 ┆ 45019        ┆ 25               │
│ 2025-04-30 00:00:00.000000 ┆ 45035        ┆ 16               │
│ 2025-05-01 00:00:00.000000 ┆ 45055        ┆ 20               │
│ 2025-05

alt.Chart(...)

In [8]:
segment("Sedmička, Lipno {")

shape: (113, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-27 00:00:00.000000 ┆ 12542        ┆ 1                │
│ 2024-12-30 00:00:00.000000 ┆ 12540        ┆ 1                │
│ 2025-01-01 00:00:00.000000 ┆ 12538        ┆ 0                │
│ 2025-01-02 00:00:00.000000 ┆ 12539        ┆ 1                │
│ 2025-01-03 00:00:00.000000 ┆ 12539        ┆ 0                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 12797        ┆ 2                │
│ 2025-04-29 00:00:00.000000 ┆ 12802        ┆ 5                │
│ 2025-04-30 00:00:00.000000 ┆ 12803        ┆ 1                │
│ 2025-05-01 00:00:00.000000 ┆ 12838        ┆ 35               │
│ 2025-05

alt.Chart(...)

In [9]:
segment("Solan last km")

shape: (123, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000 ┆ 12355        ┆ 0                │
│ 2024-12-27 00:00:00.000000 ┆ 12358        ┆ 3                │
│ 2024-12-28 00:00:00.000000 ┆ 12358        ┆ 0                │
│ 2024-12-29 00:00:00.000000 ┆ 12358        ┆ 0                │
│ 2024-12-31 00:00:00.000000 ┆ 12356        ┆ 0                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 12671        ┆ 4                │
│ 2025-04-29 00:00:00.000000 ┆ 12674        ┆ 3                │
│ 2025-04-30 00:00:00.000000 ┆ 12681        ┆ 7                │
│ 2025-05-01 00:00:00.000000 ┆ 12697        ┆ 16               │
│ 2025-05

alt.Chart(...)

In [10]:
segment("Richmond ITT1")

shape: (126, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-28 00:00:00.000000 ┆ 4618619      ┆ 402              │
│ 2024-12-29 00:00:00.000000 ┆ 4619671      ┆ 1052             │
│ 2024-12-30 00:00:00.000000 ┆ 4621281      ┆ 1610             │
│ 2024-12-31 00:00:00.000000 ┆ 4622506      ┆ 1225             │
│ 2025-01-01 00:00:00.000000 ┆ 4622622      ┆ 116              │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 4779525      ┆ 1805             │
│ 2025-04-29 00:00:00.000000 ┆ 4781923      ┆ 2398             │
│ 2025-04-30 00:00:00.000000 ┆ 4784657      ┆ 2734             │
│ 2025-05-01 00:00:00.000000 ┆ 4787128      ┆ 2471             │
│ 2025-05

alt.Chart(...)

In [11]:
segment("U Jordánu do kopce")

shape: (127, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000 ┆ 3562         ┆ 5                │
│ 2024-12-27 00:00:00.000000 ┆ 3562         ┆ 0                │
│ 2024-12-28 00:00:00.000000 ┆ 3565         ┆ 3                │
│ 2024-12-29 00:00:00.000000 ┆ 3565         ┆ 0                │
│ 2024-12-30 00:00:00.000000 ┆ 3568         ┆ 3                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 3881         ┆ 4                │
│ 2025-04-29 00:00:00.000000 ┆ 3887         ┆ 6                │
│ 2025-04-30 00:00:00.000000 ┆ 3888         ┆ 1                │
│ 2025-05-01 00:00:00.000000 ┆ 3891         ┆ 3                │
│ 2025-05

alt.Chart(...)

In [12]:
segment('400 metrů, 44 120 diváků')

shape: (126, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000 ┆ 4461         ┆ 0                │
│ 2024-12-27 00:00:00.000000 ┆ 4461         ┆ 0                │
│ 2024-12-28 00:00:00.000000 ┆ 4461         ┆ 0                │
│ 2024-12-29 00:00:00.000000 ┆ 4461         ┆ 0                │
│ 2024-12-30 00:00:00.000000 ┆ 4461         ┆ 0                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 4655         ┆ 0                │
│ 2025-04-29 00:00:00.000000 ┆ 4655         ┆ 0                │
│ 2025-04-30 00:00:00.000000 ┆ 4655         ┆ 0                │
│ 2025-05-01 00:00:00.000000 ┆ 4655         ┆ 0                │
│ 2025-05

alt.Chart(...)

In [13]:
def weekly(segment):
    return df.filter(
        pl.col('name') == segment
    ).group_by_dynamic("date", every="1d").agg(
        pl.col("effort_count").max()
    ).with_columns(
        pl.col("effort_count").diff().alias("daily_difference")
    ).with_columns(
        pl.when(pl.col("daily_difference") < 0).then(None).otherwise(pl.col("daily_difference")).alias("daily_difference")
    ).drop_nulls(
    ).with_columns(
        pl.col("date").dt.weekday().alias("day")
    ).group_by("day").agg(pl.col("daily_difference").median()).sort(by="day")

In [14]:
def graf(tyden):
    return alt.Chart(tyden).mark_bar().encode(alt.X("day:N"), alt.Y("daily_difference:Q"))

In [15]:
graf(weekly("Track Domažlice"))

alt.Chart(...)

In [16]:
graf(weekly("Velká Deštná climb "))

alt.Chart(...)

In [17]:
graf(weekly("Židlochovice -> Olympia"))

alt.Chart(...)

In [18]:
graf(weekly("Od posledního tunelu do Bílovic"))

alt.Chart(...)

In [19]:
segment("Opičí Časovka, Podolská vodárna ")

shape: (0, 3)
┌──────┬──────────────┬──────────────────┐
│ date ┆ effort_count ┆ daily_difference │
│ ---  ┆ ---          ┆ ---              │
│ str  ┆ i64          ┆ i64              │
╞══════╪══════════════╪══════════════════╡
└──────┴──────────────┴──────────────────┘


alt.Chart(...)

In [20]:
graf(weekly("Opičí Časovka, Podolská vodárna"))

alt.Chart(...)

In [21]:
segment("Vítkov up sprint ")

shape: (0, 3)
┌──────┬──────────────┬──────────────────┐
│ date ┆ effort_count ┆ daily_difference │
│ ---  ┆ ---          ┆ ---              │
│ str  ┆ i64          ┆ i64              │
╞══════╪══════════════╪══════════════════╡
└──────┴──────────────┴──────────────────┘


alt.Chart(...)

In [22]:
df.filter(pl.col("name").str.contains("Vítkov up sprint"))

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86417,8588,2024-12-27 23:22:03 CET,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86426,8589,2024-12-28 23:05:37 CET,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86438,8592,2024-12-29 23:38:26 CET,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86458,8598,2024-12-30 23:21:42 CET,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",86473,8600,2024-12-31 23:22:02 CET,0.117786
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",89797,9148,2025-04-28 23:22:04 CEST,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",89810,9153,2025-04-29 23:22:01 CEST,0.117786
14214163,"""Vítkov up sprint""","""Run""",118.2,5.1,243.2,237.2,6.0,"[50.088014, 14.449721]","[50.088404, 14.451256]","""Czechia""",null,"""Prague""",89851,9162,2025-04-30 23:05:26 CEST,0.117786


In [23]:
graf(weekly("Vítkov up sprint"))

alt.Chart(...)

In [24]:
segment("Risova studanka - od zavory")

shape: (125, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000 ┆ 41328        ┆ 2                │
│ 2024-12-26 00:00:00.000000 ┆ 41336        ┆ 8                │
│ 2024-12-27 00:00:00.000000 ┆ 41351        ┆ 15               │
│ 2024-12-28 00:00:00.000000 ┆ 41360        ┆ 9                │
│ 2024-12-29 00:00:00.000000 ┆ 41365        ┆ 5                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 42787        ┆ 31               │
│ 2025-04-29 00:00:00.000000 ┆ 42808        ┆ 21               │
│ 2025-04-30 00:00:00.000000 ┆ 42839        ┆ 31               │
│ 2025-05-01 00:00:00.000000 ┆ 42856        ┆ 17               │
│ 2025-05

alt.Chart(...)

In [25]:
segment("Sedýlko - Praděd climb")

shape: (101, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000 ┆ 31962        ┆ 0                │
│ 2024-12-26 00:00:00.000000 ┆ 31964        ┆ 2                │
│ 2024-12-27 00:00:00.000000 ┆ 31969        ┆ 5                │
│ 2024-12-28 00:00:00.000000 ┆ 31976        ┆ 7                │
│ 2024-12-29 00:00:00.000000 ┆ 31977        ┆ 1                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 32110        ┆ 5                │
│ 2025-04-29 00:00:00.000000 ┆ 32111        ┆ 1                │
│ 2025-04-30 00:00:00.000000 ┆ 32119        ┆ 8                │
│ 2025-05-01 00:00:00.000000 ┆ 32182        ┆ 63               │
│ 2025-05

alt.Chart(...)

In [26]:
segment("Malchor - Lysá Hora")

shape: (126, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000 ┆ 19801        ┆ 47               │
│ 2024-12-27 00:00:00.000000 ┆ 19835        ┆ 34               │
│ 2024-12-28 00:00:00.000000 ┆ 19870        ┆ 35               │
│ 2024-12-29 00:00:00.000000 ┆ 19910        ┆ 40               │
│ 2024-12-30 00:00:00.000000 ┆ 19952        ┆ 42               │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 21357        ┆ 6                │
│ 2025-04-29 00:00:00.000000 ┆ 21370        ┆ 13               │
│ 2025-04-30 00:00:00.000000 ┆ 21390        ┆ 20               │
│ 2025-05-01 00:00:00.000000 ┆ 21427        ┆ 37               │
│ 2025-05

alt.Chart(...)

In [27]:
graf(weekly("Tyršák čtyřstofka"))

alt.Chart(...)

In [28]:
graf(weekly('Juliska 400 m'))

alt.Chart(...)

In [29]:
graf(weekly('Karlův most ONLY'))

alt.Chart(...)

In [30]:
segment("Ořešník")

shape: (120, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000 ┆ 1105         ┆ 1                │
│ 2024-12-27 00:00:00.000000 ┆ 1105         ┆ 0                │
│ 2024-12-28 00:00:00.000000 ┆ 1106         ┆ 1                │
│ 2024-12-30 00:00:00.000000 ┆ 1105         ┆ 1                │
│ 2024-12-31 00:00:00.000000 ┆ 1106         ┆ 1                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-27 00:00:00.000000 ┆ 1154         ┆ 0                │
│ 2025-04-29 00:00:00.000000 ┆ 1153         ┆ 0                │
│ 2025-04-30 00:00:00.000000 ┆ 1153         ┆ 0                │
│ 2025-05-01 00:00:00.000000 ┆ 1156         ┆ 3                │
│ 2025-05

alt.Chart(...)

In [31]:
df = pl.read_parquet("data_raw/segments/*.parquet")

df = df.with_columns(
    (pl.col("date").cast(pl.Int64) * 1000000).cast(pl.Datetime(time_zone='CET'))
)
df = df.filter(pl.col('date').dt.hour() > 22)
df = df.sort(by="date")

In [32]:
df.columns

['id',
 'resource_state',
 'name',
 'activity_type',
 'distance',
 'average_grade',
 'maximum_grade',
 'elevation_high',
 'elevation_low',
 'start_latlng',
 'end_latlng',
 'elevation_profile',
 'elevation_profiles',
 'climb_category',
 'city',
 'state',
 'country',
 'private',
 'hazardous',
 'starred',
 'created_at',
 'updated_at',
 'total_elevation_gain',
 'map',
 'effort_count',
 'athlete_count',
 'star_count',
 'athlete_segment_stats',
 'xoms',
 'local_legend',
 'date']

In [33]:
df.group_by_dynamic("date", by="name", every="1d").agg(pl.col("effort_count").max())

C:\Users\micha\AppData\Local\Temp\ipykernel_24864\3197359101.py:1: DeprecationWarning: The argument `by` for `DataFrame.group_by_dynamic` is deprecated. It has been renamed to `group_by`.
  df.group_by_dynamic("date", by="name", every="1d").agg(pl.col("effort_count").max())


name,date,effort_count
str,"datetime[μs, CET]",i64
"""Dlouhe Strane od Loucnej""",2024-12-24 00:00:00 CET,308
"""Dlouhe Strane od Loucnej""",2024-12-25 00:00:00 CET,308
"""Dlouhe Strane od Loucnej""",2024-12-26 00:00:00 CET,308
"""Dlouhe Strane od Loucnej""",2024-12-27 00:00:00 CET,308
"""Dlouhe Strane od Loucnej""",2024-12-28 00:00:00 CET,308
…,…,…
"""kolem kapličky""",2025-04-28 00:00:00 CEST,162
"""kolem kapličky""",2025-04-29 00:00:00 CEST,166
"""kolem kapličky""",2025-04-30 00:00:00 CEST,171


In [34]:
df

id,resource_state,name,activity_type,distance,average_grade,maximum_grade,elevation_high,elevation_low,start_latlng,end_latlng,elevation_profile,elevation_profiles,climb_category,city,state,country,private,hazardous,starred,created_at,updated_at,total_elevation_gain,map,effort_count,athlete_count,star_count,athlete_segment_stats,xoms,local_legend,date
i64,i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,struct[2],i64,str,str,str,bool,bool,bool,str,str,f64,struct[3],i64,i64,i64,struct[6],struct[4],struct[7],"datetime[μs, CET]"
17521739,3,"""Dlouhe Strane od Loucnej""","""Ride""",12619.0,6.7,38.0,1333.5,492.8,"[50.069963, 17.091625]","[50.077466, 17.163214]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/OC6CGKN3U7SBXOGZOJBR6TZPDQSIOLK6LNKDEA5BVKFNGWWVWBSRKLL4SPOAGEECOMJUFTIJQ2XV5UVHNG75ABI="",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/QJQO5N7WHHPKXZIWHZIHJXGVPSY5NMSILNWETIH4KA2SFBG3ZNRVLHMDRAPHTJ6FZMKVJTDWBGKEVYGV7OWJOYI=""}",5,"""Loučná nad Desnou""","""Olomoucký kraj""","""Czechia""",false,false,true,"""2018-04-29T19:08:58Z""","""2021-05-15T02:20:58Z""",891.8,"{""s17521739"",""ghrpHseigBa@YEIU_AC[J}@Pm@Jm@BcAKRIA_@BU?]Ke@E_@O{@u@}@aAa@YWKc@[K?[So@cAQOGAEG_AyAGAUYa@y@Y[e@m@o@kBa@w@KYMGcA}BCOUi@YcBQk@]g@Uo@]u@CEGUOQg@w@_AaCEEKUa@e@[}@KOIWQ[q@w@c@o@WSMES]a@a@U[u@q@KUOc@e@q@s@m@_@_@CGQK]_@WQMUUs@OIC?QOCIEA]YYOGEM_@_@US[E[OWESe@{@a@o@OKo@q@[QY_@QMw@e@GMi@u@GOEa@Cg@?IFYFEl@Dn@JDAlABdANd@CFD`@KHB\E^FPAX@XGVBt@ERCb@?bAHj@@XDTCb@Bt@KFKVO\_@f@w@VW~@u@P]LMRc@ZgAP}@b@gAJg@Pi@Fg@Z{@\gBJ[A_@@IDYJQHM\IVSJ@r@MBETENIt@c@h@GTSFOJAv@HLCJIHM?UBKRU@SJQDsABSHkA@FPNh@NFDLV@TCJFPFDDMJh@b@Vl@`ANZ@LFP@LNd@JID@JN@F@l@Cb@@d@Fl@DHJN\LIe@?a@De@Vw@@KRu@Fs@Lo@Do@AUDa@Aa@BU?y@OeBFc@Ay@NgACi@@WF[FyAJIJ_@Hs@TkADs@Ai@O_AGyAGm@@UESCg@]aB]}@UWGY]m@Mg@Sg@E[Us@EEU?SP]PMLKBMJMLKVw@r@Od@MRERORSPMNQN]No@j@WPILUFG?WLOPE@YRGJi@Z[J]Zc@VOLk@Lk@XM@o@NY?MEw@@KAMDa@?KCE@SIMAIIKAI@KEkAUE?[GIEKMOk@Sg@Ie@ASGk@B]Eg@Fe@Ms@BWAq@Bw@A}@BEOqAAk@G_@Ag@E_@?k@MuACqA?g@C[O}AMk@Ge@[iBM]Eo@YkB?iACUB{@CO?MBq@F[EOA_@DQ@q@Hg@C_@N}BA]CQBGAUB_@P}A@_@F}@Ha@Cw@KeAWMCGMCaAz@i@X[V_@FSNE?SBYLEAM@YLg@V[Ze@RONa@Tg@JQI[CECGAOSKWSGO]iAuAWi@OMISKGS[CGOOGQE?[i@]U}@gAQMk@{@MESUWOIK]Wm@SIUECOC]W]KEGYM_@OUYKGKMWKQMKMEASMKKEMYWYQQQIO[YQIIKE?CCKBUEK?WJMC{@Ng@KSMOQSI_@w@m@g@AKa@UW?UMQGsAcAQSKUi@q@u@}A[sAOi@Km@?UFQTa@PUHOLGRUVKZWd@GX@ZGX?l@DJFLANFTABED@DA|@YJALIL@PEPBFAL@`@IL@ZALEDAD@JELBPFNK?IIc@Yo@Og@E[Ki@IMAWOo@A_@@]Co@G_@?_@I]MeBOw@C_@Qi@Ea@a@gBEk@@WTk@HIZOTGD@LCP@vAn@jBh@d@T~@VRAREj@A`@S^GZY\Kx@y@h@a@TGXShAc@tBaAbCoA`BaAv@]pBmAxFuDhAk@pBkAdAu@Z[z@sAh@m@LWDSGKEASOYEq@_@M@UOSIIISEUMCIOIGGI@KO]OGKMK_@s@EC?IQc@Ic@SUKWm@Ko@SG@SEYAOEWCGEe@CSII@MCSBGFS@IFq@AED]JEEKA_@Fe@EYEWMK?OEYIUMc@SE@USCGQKUw@Mk@Gm@Bq@Ja@D]ZaAHQx@{Cn@sB?GDGVgA^kAP[H]x@}A@IJO@IP[Ja@NSBSX]Po@JS@MJMHUTa@@UNS@QP}@Fs@FOLq@@WJe@Fw@Pu@?GHi@Dg@?a@Hw@Ia@@wCKiCQgAWcAGs@My@Co@F]Ty@Ti@\i@BIx@k@NEBELEBED?TKRQb@G^BZ@TJFCRBPHXBZRPDRVVNDHb@LRTh@VNTL?FJH@HLf@Tz@d@DHHHF@BDD?LLLBRVD?HHF@NPF`@KVEBc@MKIGCYSYGe@EG@g@K]CGCG@u@MgA?UDc@PGJIDY`@M\QVK\MdACd@Bb@AbCd@zDJf@@LJh@LJN?JCJ_@GeB?c@Ca@FiAJy@f@aARUP[ZSRWDCPB"",3}",308,214,7,"{null,null,null,null,null,0}","{""43:55"",""1:00:24"",""43:55"",{""strava://segments/17521739/leaderboard"",""overall"",""All-Time""}}","{13919744,""Kamil Filipek"",""https://dgalywyr863hv.cloudfront.net/pictures/athletes/13919744/6285929/3/large.jpg"",""1 effort in the last 90 days"",""1"",{""1 effort"",null},""strava://segments/17521739/local_legend?categories%5B%5D=overall""}",2024-12-24 23:02:03 CET
8598488,3,"""Lysa hora - jizni sjezdovka""","""Run""",298.4,21.3,36.0,1298.2,1234.6,"[49.544042, 18.452326]","[49.545601, 18.449198]","""https://d3o5xota0a1fcr.cloudfr…","{""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/5J5C33G27OKLY36XOMNHJJAEARTJAFC73RAKGNINHOEVXYUAVONOBEMXWTMUK7M2W3TVJLTWSOJPY24KN5SUUSY="",""https://d3o5xota0a1fcr.cloudfront.net/v6/charts/5CF2ETIB6LKDHWIDTGUS6ARGAUMFXUFVBHDUIUYMZLUO6Q47MANJNIOJK7U3TMDMA5UO6HNCVGGYAIALJSCONNA=""}",0,"""Ostravice""","""Moravian-Silesian

In [35]:
segment('Lužánky rovinka')

shape: (125, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000 ┆ 179488       ┆ 40               │
│ 2024-12-26 00:00:00.000000 ┆ 179527       ┆ 39               │
│ 2024-12-27 00:00:00.000000 ┆ 179560       ┆ 33               │
│ 2024-12-28 00:00:00.000000 ┆ 179584       ┆ 24               │
│ 2024-12-29 00:00:00.000000 ┆ 179626       ┆ 42               │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 190668       ┆ 267              │
│ 2025-04-29 00:00:00.000000 ┆ 190777       ┆ 109              │
│ 2025-04-30 00:00:00.000000 ┆ 190866       ┆ 89               │
│ 2025-05-01 00:00:00.000000 ┆ 190989       ┆ 123              │
│ 2025-05

alt.Chart(...)

In [36]:
segment('time trial at Svratka')

shape: (121, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000 ┆ 141661       ┆ 24               │
│ 2024-12-26 00:00:00.000000 ┆ 141680       ┆ 19               │
│ 2024-12-27 00:00:00.000000 ┆ 141706       ┆ 26               │
│ 2024-12-28 00:00:00.000000 ┆ 141721       ┆ 15               │
│ 2024-12-29 00:00:00.000000 ┆ 141735       ┆ 14               │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 146013       ┆ 79               │
│ 2025-04-29 00:00:00.000000 ┆ 146092       ┆ 79               │
│ 2025-04-30 00:00:00.000000 ┆ 146190       ┆ 98               │
│ 2025-05-01 00:00:00.000000 ┆ 146279       ┆ 89               │
│ 2025-05

alt.Chart(...)

In [37]:
segment('Od posledního tunelu do Bílovic')

shape: (118, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000 ┆ 124240       ┆ 14               │
│ 2024-12-27 00:00:00.000000 ┆ 124314       ┆ 74               │
│ 2024-12-29 00:00:00.000000 ┆ 124308       ┆ 18               │
│ 2024-12-30 00:00:00.000000 ┆ 124325       ┆ 17               │
│ 2024-12-31 00:00:00.000000 ┆ 124334       ┆ 9                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 129516       ┆ 106              │
│ 2025-04-29 00:00:00.000000 ┆ 129634       ┆ 118              │
│ 2025-04-30 00:00:00.000000 ┆ 129730       ┆ 96               │
│ 2025-05-01 00:00:00.000000 ┆ 129820       ┆ 90               │
│ 2025-05

alt.Chart(...)

In [38]:
segment('Lužánky - Plochá dráha')

shape: (117, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-27 00:00:00.000000 ┆ 32752        ┆ 2                │
│ 2024-12-29 00:00:00.000000 ┆ 32746        ┆ 0                │
│ 2024-12-30 00:00:00.000000 ┆ 32763        ┆ 17               │
│ 2024-12-31 00:00:00.000000 ┆ 32763        ┆ 0                │
│ 2025-01-02 00:00:00.000000 ┆ 32762        ┆ 5                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 34304        ┆ 0                │
│ 2025-04-29 00:00:00.000000 ┆ 34304        ┆ 0                │
│ 2025-04-30 00:00:00.000000 ┆ 34309        ┆ 5                │
│ 2025-05-01 00:00:00.000000 ┆ 34407        ┆ 98               │
│ 2025-05

alt.Chart(...)

In [39]:
segment('Risova studanka - od zavory')

shape: (125, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000 ┆ 41328        ┆ 2                │
│ 2024-12-26 00:00:00.000000 ┆ 41336        ┆ 8                │
│ 2024-12-27 00:00:00.000000 ┆ 41351        ┆ 15               │
│ 2024-12-28 00:00:00.000000 ┆ 41360        ┆ 9                │
│ 2024-12-29 00:00:00.000000 ┆ 41365        ┆ 5                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 42787        ┆ 31               │
│ 2025-04-29 00:00:00.000000 ┆ 42808        ┆ 21               │
│ 2025-04-30 00:00:00.000000 ┆ 42839        ┆ 31               │
│ 2025-05-01 00:00:00.000000 ┆ 42856        ┆ 17               │
│ 2025-05

alt.Chart(...)

In [40]:
segment('Brno Circuit')

shape: (125, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-28 00:00:00.000000 ┆ 880          ┆ 0                │
│ 2024-12-29 00:00:00.000000 ┆ 881          ┆ 1                │
│ 2024-12-30 00:00:00.000000 ┆ 881          ┆ 0                │
│ 2024-12-31 00:00:00.000000 ┆ 881          ┆ 0                │
│ 2025-01-01 00:00:00.000000 ┆ 881          ┆ 0                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-27 00:00:00.000000 ┆ 892          ┆ 0                │
│ 2025-04-28 00:00:00.000000 ┆ 892          ┆ 0                │
│ 2025-04-29 00:00:00.000000 ┆ 892          ┆ 0                │
│ 2025-05-01 00:00:00.000000 ┆ 890          ┆ 0                │
│ 2025-05

alt.Chart(...)

In [41]:
df.filter(pl.col('country').str.contains('Cze')).filter(pl.col('activity_type') == 'Run').group_by("name").agg(pl.col('effort_count').cast(int).max()).sort(by='effort_count',descending=True)

name,effort_count
str,i64
"""Stadion Na Děkance""",226958
"""Lužánky rovinka""",191096
"""Šlechtovka - finální rovinka""",168165
"""VUT horní dráha 400m""",121701
"""Strahovská dráha""",107888
…,…
"""Valy""",302
"""Autodrom Most""",236
"""kolem kapličky""",179


In [42]:
set(pl.Series(df.filter((pl.col('activity_type') == 'Run') & (pl.col('city') == 'Ostrava')).select("name")).to_list())

{'Hradní lávka => Sýkorův most',
 'Opavska down 1/2',
 'Park 1',
 'Železniční most => Jez Vítkovice'}

In [43]:
segment('Pražačka 320')

shape: (113, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-27 00:00:00.000000 ┆ 74840        ┆ 23               │
│ 2024-12-29 00:00:00.000000 ┆ 74919        ┆ 101              │
│ 2024-12-30 00:00:00.000000 ┆ 74938        ┆ 19               │
│ 2024-12-31 00:00:00.000000 ┆ 74961        ┆ 23               │
│ 2025-01-01 00:00:00.000000 ┆ 75036        ┆ 75               │
│ …                          ┆ …            ┆ …                │
│ 2025-04-26 00:00:00.000000 ┆ 79646        ┆ 158              │
│ 2025-04-27 00:00:00.000000 ┆ 79877        ┆ 231              │
│ 2025-04-28 00:00:00.000000 ┆ 80288        ┆ 411              │
│ 2025-04-30 00:00:00.000000 ┆ 80197        ┆ 192              │
│ 2025-05

alt.Chart(...)

In [44]:
segment('Park 1')

shape: (127, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-25 00:00:00.000000 ┆ 13566        ┆ 6                │
│ 2024-12-26 00:00:00.000000 ┆ 13571        ┆ 5                │
│ 2024-12-27 00:00:00.000000 ┆ 13571        ┆ 0                │
│ 2024-12-28 00:00:00.000000 ┆ 13573        ┆ 2                │
│ 2024-12-29 00:00:00.000000 ┆ 13576        ┆ 3                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 14745        ┆ 13               │
│ 2025-04-29 00:00:00.000000 ┆ 14757        ┆ 12               │
│ 2025-04-30 00:00:00.000000 ┆ 14768        ┆ 11               │
│ 2025-05-01 00:00:00.000000 ┆ 14777        ┆ 9                │
│ 2025-05

alt.Chart(...)

In [45]:
segment('Hradní lávka => Sýkorův most')

shape: (126, 3)
┌────────────────────────────┬──────────────┬──────────────────┐
│ date                       ┆ effort_count ┆ daily_difference │
│ ---                        ┆ ---          ┆ ---              │
│ str                        ┆ i64          ┆ i64              │
╞════════════════════════════╪══════════════╪══════════════════╡
│ 2024-12-26 00:00:00.000000 ┆ 13983        ┆ 6                │
│ 2024-12-27 00:00:00.000000 ┆ 13985        ┆ 2                │
│ 2024-12-28 00:00:00.000000 ┆ 13987        ┆ 2                │
│ 2024-12-29 00:00:00.000000 ┆ 13992        ┆ 5                │
│ 2024-12-30 00:00:00.000000 ┆ 13998        ┆ 6                │
│ …                          ┆ …            ┆ …                │
│ 2025-04-28 00:00:00.000000 ┆ 15039        ┆ 5                │
│ 2025-04-29 00:00:00.000000 ┆ 15054        ┆ 15               │
│ 2025-04-30 00:00:00.000000 ┆ 15058        ┆ 4                │
│ 2025-05-01 00:00:00.000000 ┆ 15073        ┆ 15               │
│ 2025-05

alt.Chart(...)

In [46]:
df.group_by_dynamic("date", by="name", every="1d").agg(pl.col("effort_count").max())

C:\Users\micha\AppData\Local\Temp\ipykernel_24864\3197359101.py:1: DeprecationWarning: The argument `by` for `DataFrame.group_by_dynamic` is deprecated. It has been renamed to `group_by`.
  df.group_by_dynamic("date", by="name", every="1d").agg(pl.col("effort_count").max())


name,date,effort_count
str,"datetime[μs, CET]",i64
"""Dlouhe Strane od Loucnej""",2024-12-24 00:00:00 CET,308
"""Dlouhe Strane od Loucnej""",2024-12-25 00:00:00 CET,308
"""Dlouhe Strane od Loucnej""",2024-12-26 00:00:00 CET,308
"""Dlouhe Strane od Loucnej""",2024-12-27 00:00:00 CET,308
"""Dlouhe Strane od Loucnej""",2024-12-28 00:00:00 CET,308
…,…,…
"""kolem kapličky""",2025-04-28 00:00:00 CEST,162
"""kolem kapličky""",2025-04-29 00:00:00 CEST,166
"""kolem kapličky""",2025-04-30 00:00:00 CEST,171
